# 05 · Persistent sessions with SQLite

Persist conversations locally using ADK's DatabaseSessionService. SQLite replaces a managed session backend for this classroom exercise.

**Runtime:** ADK 2.10.0 on  linux, (not tested with windows vs code), existing `dataengenv` kernel and ADC.

**Cost:** 1 Lite call; local SQLite. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.sessions import DatabaseSessionService
runtime = ROOT / ".runtime"
runtime.mkdir(exist_ok=True)
db_url = "sqlite+aiosqlite:///" + (runtime / "sessions.db").as_posix()
service = DatabaseSessionService(db_url=db_url)
agent = llm("persistent_shop", "Remember the user's order ID and answer briefly.")
lab = await make_lab(agent, sessions=service)
await ask(lab, "My order ID is O1001. Remember it.")
session_id = lab.session.id

# A second service object reads the same on-disk database.
reopened = DatabaseSessionService(db_url=db_url)
restored = await reopened.get_session(app_name=lab.runner.app_name, user_id=lab.user_id, session_id=session_id)
print("Restored events:", len(restored.events))
assert restored.id == session_id and len(restored.events) >= 2
await service.db_engine.dispose()
await reopened.db_engine.dispose()


## Try it

Save the session ID, restart the kernel, and load that ID from the same SQLite file.


## Reference

[Official ADK documentation](https://adk.dev/sessions/). Shared configuration: `config.json`. No environment activation or login cells are needed.
